# 🚀 Phần 5: Pipeline Tự Động Hóa Xử Lý Dữ Liệu Từ A Đến Z (Comprehensive End-to-End Pipeline)

Trong môi trường sản xuất (Production / ML Pipeline), việc viết code rời rạc sẽ khó bảo trì và dễ gây lỗi. Chúng ta cần đóng gói toàn bộ quy trình tiền xử lý dữ liệu vào một **Class / Function Pipeline** có khả năng:
1. Nhận vào dữ liệu thô (Raw Data).
2. Tự động khử trùng lặp (Deduplication).
3. Chuẩn hóa văn bản và định danh (Text & Category Normalization).
4. Ép kiểu chuẩn xác (Data Types & Datetime Parsing).
5. Xử lý dữ liệu khuyết thiếu thông minh (Smart Imputation).
6. Xử lý giá trị ngoại lai (Outlier Handling).
7. Tạo các đặc trưng mới (Feature Engineering: Doanh thu thực tế, ngày trong tuần...).
8. Xuất báo cáo chất lượng dữ liệu và lưu file kết quả.

---


In [ ]:
import pandas as pd
import numpy as np
import re

class DataCleaningPipeline:
    """
    Pipeline tự động làm sạch và tiền xử lý dữ liệu đơn hàng khách hàng.
    """
    def __init__(self, raw_filepath="data/customer_orders_raw.csv"):
        self.raw_filepath = raw_filepath
        self.raw_df = None
        self.cleaned_df = None
        
        # Bảng chuẩn hóa địa danh
        self.city_mapping = {
            'ha noi': 'Ha Noi',
            'hà nội': 'Ha Noi',
            'hcm': 'Ho Chi Minh',
            'ho chi minh': 'Ho Chi Minh',
            'da nang': 'Da Nang',
            'hai phong': 'Hai Phong',
            'can tho': 'Can Tho',
            'cần thơ': 'Can Tho',
            'hue': 'Hue',
            'quang ninh': 'Quang Ninh',
            'nha trang': 'Nha Trang'
        }
        
    def load_data(self):
        """Đọc dữ liệu thô ban đầu"""
        self.raw_df = pd.read_csv(self.raw_filepath)
        print(f"[1/7] Đã tải dữ liệu thô: {self.raw_df.shape[0]} dòng, {self.raw_df.shape[1]} cột")
        return self.raw_df.copy()

    def remove_duplicates(self, df):
        """Khử dữ liệu trùng lặp"""
        initial_count = len(df)
        # Khử trùng lặp theo order_id
        df = df.drop_duplicates(subset=['order_id'], keep='first').copy()
        print(f"[2/7] Khử trùng lặp: Đã loại bỏ {initial_count - len(df)} dòng")
        return df

    def clean_text_and_categories(self, df):
        """Chuẩn hóa chuỗi văn bản và danh mục"""
        # Chuẩn hóa tên khách hàng
        df['customer_name'] = (
            df['customer_name']
            .astype(str)
            .str.strip()
            .str.replace(r'\s+', ' ', regex=True)
            .str.title()
        )
        
        # Chuẩn hóa email
        df['email'] = df['email'].astype(str).str.strip().str.lower()
        email_regex = r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$'
        df['is_valid_email'] = df['email'].str.match(email_regex)
        df.loc[~df['is_valid_email'], 'email'] = np.nan
        df.drop(columns=['is_valid_email'], inplace=True)
        
        # Chuẩn hóa tên thành phố
        city_raw = df['city'].astype(str).str.strip().str.lower()
        df['city'] = city_raw.map(self.city_mapping).fillna('Other')
        
        # Chuẩn hóa danh mục sản phẩm và trạng thái
        df['product_category'] = df['product_category'].astype(str).str.strip().str.title().astype('category')
        df['payment_status'] = df['payment_status'].astype(str).str.strip().str.upper().astype('category')
        
        print("[3/7] Hoàn thành chuẩn hóa văn bản & danh mục")
        return df

    def parse_data_types(self, df):
        """Chuyển đổi và ép kiểu dữ liệu chuẩn"""
        # 1. Price: Bỏ '$' và ','
        price_clean = df['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
        df['price'] = pd.to_numeric(price_clean, errors='coerce')
        
        # 2. Discount Rate: Chuyển về float [0.0, 1.0]
        disc_clean = df['discount_rate'].astype(str).str.replace('%', '', regex=False).str.strip()
        df['discount_rate'] = pd.to_numeric(disc_clean, errors='coerce') / 100.0
        df['discount_rate'] = df['discount_rate'].fillna(0.0)
        
        # 3. Quantity: Chuyển sang số
        df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
        
        # 4. Phone: Chuẩn hóa định dạng số điện thoại chuỗi
        phone_digits = df['phone'].astype(str).str.replace(r'[^0-9]', '', regex=True)
        phone_fixed = phone_digits.apply(lambda x: ('0' + x) if (len(x) == 9 and not x.startswith('0')) else x)
        df['phone'] = phone_fixed.replace(['', '0nan', 'nan'], np.nan).fillna('Unknown')
        
        # 5. Order Date: Chuyển sang Datetime
        df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')
        
        print("[4/7] Hoàn thành ép kiểu dữ liệu số, ngày tháng, danh mục")
        return df

    def handle_missing_and_outliers(self, df):
        """Xử lý giá trị khuyết thiếu và ngoại lai"""
        # Xử lý Age Outliers & Missing: hợp lệ 10-100 tuổi, còn lại impute bằng median
        age_num = pd.to_numeric(df['age'], errors='coerce')
        median_age = age_num[(age_num >= 10) & (age_num <= 100)].median()
        df['age'] = age_num.apply(lambda x: x if (10 <= x <= 100) else median_age).astype('int64')
        
        # Xử lý Quantity Outliers: clip trong khoảng [1, 10]
        median_qty = df.loc[(df['quantity'] > 0) & (df['quantity'] <= 10), 'quantity'].median()
        df['quantity'] = df['quantity'].apply(lambda x: x if (1 <= x <= 10) else median_qty).astype('int64')
        
        # Xử lý Price Missing: điền median của category đó
        df['price'] = df.groupby('product_category', observed=False)['price'].transform(
            lambda grp: grp.fillna(grp.median() if not grp.median() != grp.median() else 100.0)
        )
        
        # Nếu ngày đặt hàng bị rỗng, điền bằng ngày gần nhất phía trước (ffill) hoặc ngày mặc định
        df['order_date'] = df['order_date'].ffill().bfill()
        
        print("[5/7] Hoàn thành xử lý missing values & ngoại lai")
        return df

    def engineer_features(self, df):
        """Tạo các thuộc tính dẫn xuất hữu ích"""
        # Doanh thu thực tế = Giá * Số lượng * (1 - Tỉ lệ giảm giá)
        df['total_amount'] = (df['price'] * df['quantity'] * (1.0 - df['discount_rate'])).round(2)
        
        # Đặc trưng thời gian
        df['order_year'] = df['order_date'].dt.year
        df['order_month'] = df['order_date'].dt.month
        df['order_day_name'] = df['order_date'].dt.day_name()
        df['is_weekend'] = df['order_date'].dt.dayofweek.isin([5, 6]).astype(int)
        
        print("[6/7] Hoàn thành trích xuất đặc trưng mới (Feature Engineering)")
        return df

    def run_pipeline(self, output_path="data/customer_orders_cleaned.csv"):
        """Chạy toàn bộ pipeline từ đầu đến cuối"""
        print("=== BẮT ĐẦU CHẠY PIPELINE LÀM SẠCH DỮ LIỆU ===")
        df = self.load_data()
        df = self.remove_duplicates(df)
        df = self.clean_text_and_categories(df)
        df = self.parse_data_types(df)
        df = self.handle_missing_and_outliers(df)
        df = self.engineer_features(df)
        
        df.reset_index(drop=True, inplace=True)
        self.cleaned_df = df
        
        if output_path:
            df.to_csv(output_path, index=False)
            print(f"[7/7] Đã lưu dataset sạch vào: {output_path}")
            
        print("=== HOÀN THÀNH PIPELINE THÀNH CÔNG ===\n")
        return self.cleaned_df


## 1. Thực Thi Toàn Bộ Pipeline
Hãy khởi tạo đối tượng `DataCleaningPipeline` và quan sát tiến trình thực thi.


In [ ]:
# Khởi tạo và chạy pipeline
pipeline = DataCleaningPipeline(raw_filepath="data/customer_orders_raw.csv")
cleaned_df = pipeline.run_pipeline(output_path="data/customer_orders_cleaned.csv")
cleaned_df.head(10)


## 2. Báo Cáo Đối Chiếu Chất Lượng Dữ Liệu (Before vs After Quality Audit)
So sánh dữ liệu trước và sau khi xử lý để đảm bảo dữ liệu đạt tiêu chuẩn chất lượng cao nhất.


In [ ]:
print("=== BÁO CÁO ĐỐI CHIẾU CHẤT LƯỢNG DỮ LIỆU ===")
print(f"- Kích thước ban đầu: {pipeline.raw_df.shape}")
print(f"- Kích thước sau làm sạch: {cleaned_df.shape}")
print(f"- Số lượng Missing Values ban đầu: {pipeline.raw_df.isna().sum().sum()}")
print(f"- Số lượng Missing Values sau làm sạch: {cleaned_df.isna().sum().sum()}")
print(f"- Tổng doanh thu sau làm sạch: ${cleaned_df['total_amount'].sum():,.2f}")

print("\nKiểu dữ liệu sau khi làm sạch:")
print(cleaned_df.dtypes)


## 3. Tổng Kết & Kiến Trúc Sản Xuất (Production Best Practices)
1. **Tính Module hóa (Modularity)**: Tách riêng từng bước xử lý giúp dễ dàng kiểm thử đơn vị (Unit Test) và mở rộng.
2. **Tính Tái Sử Dụng (Reusability)**: Pipeline có thể áp dụng trực tiếp cho các lô dữ liệu mới (Batch Data) hoặc dữ liệu Stream trong thực tế.
3. **Tính Bền Vững (Robustness)**: Luôn dự phòng các trường hợp giá trị lỗi bất ngờ với `errors='coerce'` và các giá trị mặc định an toàn.
